This notebook combines participant information, activity data, physiological signals, stress measurements, and ActiGraph-derived MVPA into a daily dataset used as input for the finite state machine

In [ ]:
import pandas as pd
import numpy as np
import re

Load config

In [ ]:
%run 00_config.ipynb

Participant Profile Classification

In [ ]:
participants = pd.read_excel(PARTICIPANTS_FILE)

# Keep variables required for profile classification
participants = participants[[
    "user_id",
    "bmi",
    "bmi_class",
    "accelerometer"
]].copy()

participants["user_id"] = participants["user_id"].astype(int)
participants["accelerometer"] = pd.to_numeric(participants["accelerometer"], errors="coerce")

def classify_bmi_profile(bmi_class):
    # Assign participants to the BMI profiles used by the state machine
    bmi_class = str(bmi_class).lower()

    if "obes" in bmi_class or "excesso" in bmi_class or "overweight" in bmi_class:
        return "overweight_obese"

    return "healthy"

participants["bmi_profile"] = participants["bmi_class"].apply(classify_bmi_profile)

participants.head()

Get activity features (hr and rmssd) and median

In [ ]:
activity = pd.read_csv(ACTIVITY_FILE)

activity["timestamp"] = pd.to_datetime(activity["timestamp"])
activity["date"] = activity["timestamp"].dt.date

# Calculate daily median heart rate and RMSSD for each participant
daily_physio = (
    activity
    .groupby(["user_id", "date"])
    .agg(
        hr_median=("hr", "median"),
        rmssd_median=("rmssd", "median")
    )
    .reset_index()
)

daily_physio.head()

Daily Step Count (Read steps)

In [ ]:
daily_activity = pd.read_csv(DAILY_ACTIVITY_FILE)

daily_activity["date"] = pd.to_datetime(daily_activity["day"]).dt.date

daily_activity = daily_activity.rename(columns={
    "steps_diff": "steps"
})

daily_steps = daily_activity[[
    "user_id",
    "date",
    "steps"
]].copy()

daily_steps.head()

Read Mood file

In [ ]:
mood = pd.read_csv(MOOD_FILE)

mood["timestamp"] = pd.to_datetime(mood["timestamp"])
mood["date"] = mood["timestamp"].dt.date

# Calculate mean daily stress for each participant
daily_stress = (
    mood
    .groupby(["user_id", "date"])
    .agg(
        avg_stress=("stress_value", "mean")
    )
    .reset_index()
)

daily_stress.head()

Read MVPA from actigraph (get daily MVPA)

In [ ]:
actigraph = pd.read_excel(ACTIGRAPH_FILE, sheet_name="Daily")

def extract_accelerometer_id(filename):
    # Extract the accelerometer ID from the filename
    match = re.search(r"MOS2E(\d+)", str(filename))

    if match:
        return int(match.group(1)[-3:])

    return np.nan


actigraph["accelerometer"] = actigraph["Filename"].apply(extract_accelerometer_id)

actigraph_daily = actigraph[[
    "accelerometer",
    "Date",
    "Total MVPA"
]].copy()

actigraph_daily = actigraph_daily.rename(columns={
    "Date": "date",
    "Total MVPA": "mvpa_minutes"
})

actigraph_daily["date"] = pd.to_datetime(actigraph_daily["date"]).dt.date

actigraph_daily.head()

Conect acceleromete to user id

In [ ]:
actigraph_daily = actigraph_daily.merge(
    participants[["user_id", "accelerometer"]],
    on="accelerometer",
    how="left"
)

actigraph_daily = actigraph_daily.dropna(subset=["user_id"])

actigraph_daily["user_id"] = actigraph_daily["user_id"].astype(int)

actigraph_daily.head()

Build daily dataset with all the information above

In [ ]:
# Merge daily step and physiological features
daily_dataset = daily_steps.merge(
    daily_physio,
    on=["user_id", "date"],
    how="outer"
)

# Add daily stress
daily_dataset = daily_dataset.merge(
    daily_stress,
    on=["user_id", "date"],
    how="outer"
)

# Add daily MVPA
daily_dataset = daily_dataset.merge(
    actigraph_daily[["user_id", "date", "mvpa_minutes"]],
    on=["user_id", "date"],
    how="outer"
)

# Add participant BMI information and profile
daily_dataset = daily_dataset.merge(
    participants[["user_id", "bmi", "bmi_class", "bmi_profile"]],
    on="user_id",
    how="left"
)

# Sort by participant id and date
daily_dataset = daily_dataset.sort_values(
    ["user_id", "date"]
).reset_index(drop=True)

daily_dataset.head()

Handling missing MVPA records: Missing MVPA values are interpreted as zero minutes of MVPA. Rows containing no information in any of the other relevant daily variables are then removed.

In [ ]:
# Treat missing MVPA records as zero minutes
daily_dataset["mvpa_minutes"] = (
    daily_dataset["mvpa_minutes"]
    .fillna(0)
)

# Identify rows where MVPA is zero and all other
# relevant daily variables are missing
only_mvpa_zero = (
    daily_dataset["mvpa_minutes"].eq(0)
    & daily_dataset["steps"].isna()
    & daily_dataset["hr_median"].isna()
    & daily_dataset["rmssd_median"].isna()
    & daily_dataset["avg_stress"].isna()
)

# Remove rows without meaningful daily information
daily_dataset = daily_dataset[
    ~only_mvpa_zero
].copy()

Imopute missing steps and stress values using participant-specific statistics. If a participant has no available history for a given variable, the corresponding global statistic is used as a fallback.

In [ ]:
# =========================
# GLOBAL FALLBACK VALUES
# =========================

global_steps_median = daily_dataset["steps"].median()
global_stress_mean = daily_dataset["avg_stress"].mean()


# =========================
# PARTICIPANT-SPECIFIC VALUES
# =========================

user_imputation_values = (
    daily_dataset
    .groupby("user_id")
    .agg(
        steps_user_median=("steps", "median"),
        stress_user_mean=("avg_stress", "mean")
    )
    .reset_index()
)

# Use global values when no participant-specific history is available
user_imputation_values["steps_user_median"] = (
    user_imputation_values["steps_user_median"]
    .fillna(global_steps_median)
)

user_imputation_values["stress_user_mean"] = (
    user_imputation_values["stress_user_mean"]
    .fillna(global_stress_mean)
)


# =========================
# APPLY IMPUTATION
# =========================

daily_dataset = daily_dataset.merge(
    user_imputation_values,
    on="user_id",
    how="left"
)

daily_dataset["steps"] = (
    daily_dataset["steps"]
    .fillna(daily_dataset["steps_user_median"])
)

daily_dataset["avg_stress"] = (
    daily_dataset["avg_stress"]
    .fillna(daily_dataset["stress_user_mean"])
)


# Remove temporary imputation columns
daily_dataset = daily_dataset.drop(
    columns=[
        "steps_user_median",
        "stress_user_mean"
    ]
)

daily_dataset.head()

Imput missing HR and RMSSD values with user baseline. If a participant has no available physiological history, global median values are used as fallback values.

In [ ]:
# =========================
# GLOBAL PHYSIOLOGICAL FALLBACKS
# =========================

global_hr_median = daily_dataset["hr_median"].median()
global_rmssd_median = daily_dataset["rmssd_median"].median()


# =========================
# PARTICIPANT-SPECIFIC BASELINES
# =========================

user_baselines = (
    daily_dataset
    .groupby("user_id")
    .agg(
        hr_baseline=("hr_median", "median"),
        rmssd_baseline=("rmssd_median", "median")
    )
    .reset_index()
)

# Use global medians when no participant-specific history is available
user_baselines["hr_baseline"] = (
    user_baselines["hr_baseline"]
    .fillna(global_hr_median)
)

user_baselines["rmssd_baseline"] = (
    user_baselines["rmssd_baseline"]
    .fillna(global_rmssd_median)
)


# =========================
# MERGE BASELINES
# =========================

daily_dataset = daily_dataset.merge(
    user_baselines,
    on="user_id",
    how="left"
)


# =========================
# IDENTIFY DAYS WITH ACTIVITY DATA
# =========================

activity_present = (
    daily_dataset["steps"].notna()
    | (daily_dataset["mvpa_minutes"] > 0)
)


# =========================
# IMPUTE HR
# =========================

hr_missing = (
    activity_present
    & daily_dataset["hr_median"].isna()
)

daily_dataset.loc[
    hr_missing,
    "hr_median"
] = daily_dataset.loc[
    hr_missing,
    "hr_baseline"
]


# =========================
# IMPUTE RMSSD
# =========================

rmssd_missing = (
    activity_present
    & daily_dataset["rmssd_median"].isna()
)

daily_dataset.loc[
    rmssd_missing,
    "rmssd_median"
] = daily_dataset.loc[
    rmssd_missing,
    "rmssd_baseline"
]


# Remove temporary baseline columns
daily_dataset = daily_dataset.drop(
    columns=[
        "hr_baseline",
        "rmssd_baseline"
    ]
)

daily_dataset.head()

Define valid days (if there are all the features available for a certain day). Activity days require step and MVPA data, stress days require HR, RMSSD, and stress data, and complete days require both sets of variables.

In [ ]:
# Valid activity day:
# both steps and MVPA must be available
daily_dataset["valid_activity_day"] = (
    daily_dataset[
        ["steps", "mvpa_minutes"]
    ]
    .notna()
    .all(axis=1)
)

# Valid stress day:
# HR, RMSSD, and stress must all be available
daily_dataset["valid_stress_day"] = (
    daily_dataset[
        ["hr_median", "rmssd_median", "avg_stress"]
    ]
    .notna()
    .all(axis=1)
)

# Complete valid day:
# both activity and stress criteria must be satisfied
daily_dataset["valid_complete_day"] = (
    daily_dataset["valid_activity_day"]
    & daily_dataset["valid_stress_day"]
)

daily_dataset.head()

Count valid days per user

In [ ]:
valid_days_summary = (
    daily_dataset
    .groupby("user_id")
    .agg(
        total_days=("date", "nunique"),
        activity_days=("valid_activity_day", "sum"),
        stress_days=("valid_stress_day", "sum"),
        complete_days=("valid_complete_day", "sum"),
        start_date=("date", "min"),
        end_date=("date", "max"),
        bmi_profile=("bmi_profile", "first")
    )
    .reset_index()
    .sort_values("complete_days", ascending=False)
)

valid_days_summary

Save outputs: The complete daily multimodal dataset and the participant-level valid-day summary are saved for use in the finite state machine analysis.

In [ ]:
daily_dataset.to_excel(
    OUTPUT_DIR / "daily_dataset.xlsx",
    index=False
)

valid_days_summary.to_excel(
    OUTPUT_DIR / "valid_days_summary.xlsx",
    index=False
)

print("Saved:")
print(OUTPUT_DIR / "daily_dataset.xlsx")
print(OUTPUT_DIR / "valid_days_summary.xlsx")

print("Daily dataset shape:", daily_dataset.shape)